# E v8：75 条已定案 NQ Train 开发样本

选择 Kaggle T4 GPU，打开联网，添加配套 tar.gz 为输入，依次运行全部单元。
也支持 Colab T4 上传。四组：v7 对照、v8 完整流程、去原答案核验、去候选核验。
每组同 75 条，共 300 条输出，最多 600 次 Qwen 生成。C1 直接复用原始结果，零新增调用。
11 条 E 目标与 64 条正确对照只用于开发；25 条待定不在包内。同模型核验不是独立裁判。
结束后下载导出的 ZIP；语义收益留待人工审核，不由 EM 或核验通过自动认定。


In [ ]:
import json, os, sys, subprocess, hashlib, tarfile, shutil
from pathlib import Path
import torch
assert sys.version_info >= (3, 11), 'Python 3.11+ required'
assert torch.cuda.is_available(), 'Select a GPU runtime'
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
EXPECTED_SHA256 = '906f096f78e36a4f272cdd863ce2e7383d62acb638c1e480a06c5d3f75daa7a3'
BUNDLE_NAME = 'answer_repair_v8_nq75_gpu'
OUTPUT = WORK / (BUNDLE_NAME.removesuffix('_gpu') + '_outputs')
OUTPUT.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0), 'CUDA:', torch.version.cuda)


In [ ]:
archive = WORK / (BUNDLE_NAME + '.tar.gz')
if not archive.exists() and Path('/kaggle/input').exists():
    matches = list(Path('/kaggle/input').rglob(archive.name))
    assert len(matches) == 1, 'Attach exactly one matching tar.gz input'
    archive = matches[0]
elif not archive.exists():
    from google.colab import files
    uploaded = files.upload()
    assert archive.name in uploaded, 'Upload the matching tar.gz'
    archive.write_bytes(uploaded[archive.name])
assert hashlib.sha256(archive.read_bytes()).hexdigest() == EXPECTED_SHA256, 'Archive hash mismatch'
extract_root = WORK / ('e_v8_' + EXPECTED_SHA256[:12])
BUNDLE = extract_root / BUNDLE_NAME
if not BUNDLE.exists():
    extract_root.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive) as handle:
        handle.extractall(extract_root, filter='data')
checksums = json.loads((BUNDLE / 'bundle_checksums.json').read_text())
assert all(hashlib.sha256((BUNDLE / p).read_bytes()).hexdigest() == h for p, h in checksums.items())
manifest = json.loads((BUNDLE / 'manifest.json').read_text())
assert manifest['samples'] == 75
for source in ('manifest.json', 'bundle_checksums.json'):
    shutil.copy2(BUNDLE / source, OUTPUT / source)
shutil.copy2(archive, OUTPUT / archive.name)
print('Bundle verified:', len(checksums), 'files;', manifest['samples'], 'samples')


In [ ]:
def run_logged(argv, name):
    with (OUTPUT / name).open('a', encoding='utf-8') as log:
        log.write('COMMAND ' + json.dumps(argv) + '\n'); log.flush()
        process = subprocess.Popen(argv, cwd=BUNDLE, env={**os.environ, 'PYTHONUNBUFFERED': '1'},
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            status = process.wait()
        except BaseException:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
    if status: raise subprocess.CalledProcessError(status, argv)

run_logged([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-answer-repair-v8.txt'], 'install.log')
run_logged([sys.executable, '-m', 'pip', 'freeze'], 'packages.log')
run_logged([sys.executable, 'scripts/check_answer_repair_environment.py', '--require-cuda',
            '--out', str(OUTPUT / 'environment.json')], 'environment.log')
run_logged([sys.executable, '-m', 'unittest', *manifest['portable_tests']], 'tests.log')
RUNS = []
for arm in manifest['protocol']:
    strategy = arm['strategy']
    argv = [sys.executable, '-u', 'scripts/run_answer_repair.py', '--dataset', 'nq',
            '--c1', str(BUNDLE / 'inputs/nq75_c1.jsonl'), '--all', '--strategy', strategy,
            '--max-new-tokens', str(arm['max_new_tokens']), '--out', str(OUTPUT / (strategy + '.jsonl'))]
    RUNS.append((strategy, argv))
    run_logged(argv + ['--check-only'], strategy + '_preflight.log')


依次完成四组，不根据中途结果改提示或挑选样本。
中断后可在相同环境续跑；已完成的预测会校验并跳过。更换环境需保留旧输出，另开新实验目录。


In [ ]:
try:
    for strategy, argv in RUNS:
        run_logged(argv, strategy + '_run.log')
finally:
    # Preserve partial progress even if a backend fails; errors still stop the experiment.
    partial = shutil.make_archive(str(WORK / (OUTPUT.name + '_partial')), 'zip', OUTPUT)
    print('Recovery archive:', partial)


In [ ]:
run_logged([sys.executable, 'scripts/summarize_answer_repair_v8.py',
            '--bundle', str(BUNDLE), '--results', str(OUTPUT)], 'comparison.log')
report = json.loads((OUTPUT / 'comparison.json').read_text())
assert report['all_arms_complete'], 'Incomplete experiment; preserve partial output'
export = shutil.make_archive(str(WORK / (OUTPUT.name + '_export')), 'zip', OUTPUT)
print('Result archive:', export)
print('SHA256:', hashlib.sha256(Path(export).read_bytes()).hexdigest())
print('Changed outputs pending review:', report['pending_changed_outputs'])
if not Path('/kaggle/working').exists():
    from google.colab import files
    files.download(export)
